# Legal Obligation Extraction with LLMs

This notebook runs prompting experiments for extracting structured
obligation tuples from regulatory text.

Model used in this notebook:
AdaptLaw / AdaptLLM Law-Chat

In [ ]:
!pip install --upgrade accelerate transformers bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from openpyxl import load_workbook

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================================
# Configuration
# ========================================
MODEL_NAME = "AdaptLLM/law-chat"
TEMPERATURE = 0.0  # deterministic




In [ ]:
EXCEL_FILE = "Legal_Text_Extraction.xlsx"
SHEET_NAME = "New Text"
INPUT_COLUMN = "D"
OUTPUT_COLUMN = "E"

In [ ]:
!pip install -q accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ========================================
# Load tokenizer and model (4-bit for Colab)
# ========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config, # Use the quantization_config
    dtype= torch.float16
)

In [ ]:
# ========================================
# Few-shot and system prompts (unchanged)
# ========================================
# ========== Prompt Builder ==========
FEW_SHOT_EXAMPLES = """
Example 1
Input:
"(b) Electronic records.
Persons who use electronic systems to create, modify, maintain, archive, retrieve, or transmit records required under this chapter must ensure that the systems used are capable of generating accurate and complete copies of records in both human readable and electronic form suitable for inspection, review, and copying by the Food and Drug Administration. Persons shall implement procedures to protect records to enable their accurate and ready retrieval throughout the records retention period."

Expected output:
1. Actor: Persons who use electronic systems to create, modify, maintain, archive, retrieve, or transmit records
   Action: Ensure
   Object: systems used are capable of generating accurate and complete copies of records
   Condition: suitable for inspection, review, and copying by the Food and Drug Administration

2. Actor: Persons
   Action: Implement
   Object: procedures to protect records
   Condition: to enable accurate and ready retrieval throughout the records retention period


Example 2
Input:
"(c) Submission of information.
Any person submitting information to the Food and Drug Administration shall ensure that the submission is truthful and not misleading in any particular.
The submitter must retain records supporting the submission and must make such records available to the agency upon request."

Expected output:
1. Actor: Any person submitting information to the Food and Drug Administration
   Action: Ensure
   Object: submission is truthful and not misleading
   Condition: None

2. Actor: The submitter
   Action: Retain
   Object: records supporting the submission
   Condition: None

3. Actor: The submitter
   Action: Make available
   Object: records supporting the submission to the agency
   Condition: upon request


Example 3
Input:
"(d) Inspection and access to records.
Persons subject to this chapter shall permit authorized employees of the Food and Drug Administration, at reasonable times and in a reasonable manner, to have access to and copy records required to be maintained under this chapter. Such persons must provide reasonable assistance to facilitate the inspection of such records."

Expected output:
1. Actor: Persons subject to this chapter
   Action: Permit
   Object: authorized employees of the Food and Drug Administration to access and copy records required to be maintained
   Condition: at reasonable times and in a reasonable manner

2. Actor: Such persons
   Action: Provide
   Object: reasonable assistance to facilitate inspection of records
   Condition: None
""".strip()


def extraction_prompt(excerpt: str) -> str:
    return f"""
Task:
Extract all **obligatory regulatory requirements** from the following regulatory text.

Definition of an obligatory regulation:
A sentence that imposes a **mandatory duty, responsibility, or required action** on an actor.

Obligations often contain language such as:
- must
- shall
- required to
- responsible for
- must ensure
- must include
- must maintain
- must implement
- is responsible for
- is required
- must be
- must contain

However, the wording may vary. Extract any statement that clearly imposes a **mandatory regulatory requirement**.

Instructions:
1. Identify sentences that impose mandatory obligations.
2. Do NOT infer obligations that are not stated.
3. Ignore recommendations such as "may", "should", or "recommend".
4. Split compound obligations into separate entries when possible.
5. For each obligation extract:
   - Actor
   - Action (short verb phrase)
   - Object (what the action applies to)
   - Condition (if applicable)
6. If no condition exists, write "None".

Output format:
Numbered list using these labels exactly:
Actor:
Action:
Object:
Condition:

Few-shot examples:
{FEW_SHOT_EXAMPLES}

Now extract obligations from this input.

Input:
{excerpt}

Expected output:
""".strip()

# ========== System Prompt ==========
system_prompt = """
You are an assistant specialized in regulatory compliance extraction.

Your task is to identify and extract **obligatory regulatory requirements**.

Definition: An obligation is any statement that imposes a mandatory duty, responsibility, or required action on an actor.

Obligation language may include, but is not limited to:
must, shall, required to, responsible for, must ensure, must include, must maintain, must implement, is responsible for, must be, must contain.

Do NOT extract:
- recommendations (may, should, recommend)
- explanations, definitions, or examples
- obligations not explicitly stated

CFR-style hierarchical numbering:
- Recognize sections like (a), (1), (i), (A)
- Do NOT treat list items as separate obligations unless they contain mandatory language
- If a parent clause contains an obligation, treat child list items as objects

For each obligation return:
Actor
Action
Object
Condition

Return only the numbered list of obligations or:
"No explicit obligations found."
""".strip()


In [ ]:
# ========================================
# Model inference
# ========================================
def extract_obligations(excerpt: str) -> str:
    user_prompt = extraction_prompt(excerpt)
    prompt_few_shot = f"<s>[INST] <<SYS>>{system_prompt}<</SYS>>\n\n{user_prompt} [/INST]"

    inputs_few_shot = tokenizer(prompt_few_shot, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    outputs_few_shot = model.generate(input_ids=inputs_few_shot, max_length=4096, temperature=TEMPERATURE)[0]

    answer_start_fewshot = int(inputs_few_shot.shape[-1])
    pred_fewshot = tokenizer.decode(outputs_few_shot[answer_start_fewshot:], skip_special_tokens=True)

    return pred_fewshot.strip()


In [ ]:
# ========================================
# Excel processing
# ========================================
def detect_start_row(worksheet, input_column: str) -> int:
    first_value = worksheet[f"{input_column}1"].value
    if isinstance(first_value, str) and first_value.strip().lower() in {"excerpt", "excerpts", "text", "input"}:
        return 2
    return 1

def process_excel_file():
    workbook = load_workbook(EXCEL_FILE)
    worksheet = workbook[SHEET_NAME] if SHEET_NAME else workbook.active
    start_row = detect_start_row(worksheet, INPUT_COLUMN)

    processed_rows = 0
    skipped_rows = 0

    for row_idx in range(start_row, worksheet.max_row + 1):
        excerpt = worksheet[f"{INPUT_COLUMN}{row_idx}"].value
        excerpt_str = str(excerpt).strip() if excerpt else ""

        if not excerpt_str:
            skipped_rows += 1
            continue

        try:
            response_text = extract_obligations(excerpt_str)
        except Exception as e:
            response_text = f"ERROR: {e}"

        worksheet[f"{OUTPUT_COLUMN}{row_idx}"] = response_text
        processed_rows += 1
        print(f"Processed row {row_idx} -> {OUTPUT_COLUMN}")

    workbook.save(EXCEL_FILE)
    print(f"\nSaved {processed_rows} outputs to column {OUTPUT_COLUMN}")
    print(f"Skipped {skipped_rows} empty rows from column {INPUT_COLUMN}")


In [ ]:
process_excel_file()